# ARC-v0.16.1 — Deployable Predictor Feature-Ablation Audit

This notebook addresses the remaining reviewer-facing question:

> **How much of the deployable ARC-v0.15 predictor comes from policy variables alone, and how much additional information comes from query-specific PQ32 observables?**

The audit preserves the ARC-v0.15 target and classifier family and compares:

1. **alpha only**
2. **policy only** — alpha, log(k), feedback family, temperature
3. **PQ32 query statistics only** — PQ32 entropy and PQ32 margin
4. **full deployable** — PQ32 query statistics + policy variables

Primary evaluation is performed once on the untouched ARC-v0.13 FEVER validation split.

This is a **post-hoc predictor-ablation audit**, not a new preregistered primary experiment. It does not rerun retrieval and does not access FEVER test data.


In [ ]:
# ============================================================
# Cell 1 — Imports / Drive / source integrity
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import gc
import warnings

import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260816
EPS = 0.002
BOOTSTRAP_REPS = 2000

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed"

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
assert ARC_ROOT.is_dir(), ARC_ROOT

V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"
assert V013_ROOT.is_dir(), V013_ROOT

PREFERRED_V013_RUN = V013_ROOT / "20260817-140640"

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def is_complete_v013_run(path):
    return (
        path.is_dir()
        and len(list(path.glob("fit-*.parquet"))) == 44
        and len(list(path.glob("validation-*.parquet"))) == 44
        and (path / "v013_fever_boundary_protocol.json").is_file()
        and (path / "v013_validation_continuation_report.json").is_file()
    )

if is_complete_v013_run(PREFERRED_V013_RUN):
    V013_RUN = PREFERRED_V013_RUN
else:
    candidates = sorted(
        [p for p in V013_ROOT.iterdir() if p.is_dir() and is_complete_v013_run(p)],
        reverse=True,
    )
    assert candidates, "No complete ARC-v0.13 run found"
    V013_RUN = candidates[0]

PROTOCOL_PATH = V013_RUN / "v013_fever_boundary_protocol.json"
VALIDATION_REPORT_PATH = V013_RUN / "v013_validation_continuation_report.json"

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))

assert protocol["status"] == "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP"
assert validation_report["status"] == "ARC_V013_FEVER_VALIDATION_CONTINUATION_COMPLETE"
assert protocol["test_access_allowed"] is False
assert validation_report["test_accessed"] is False
assert np.isclose(float(protocol["regime_threshold_abs_slope"]), EPS)

OUT_ROOT = ARC_ROOT / "deployable-predictor-ablation-v0161"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

print("Source v0.13:", V013_RUN)
print("Frozen epsilon:", EPS)
print("Output:", OUT)
print("ARC-v0.16.1 PREFLIGHT — PASS")


In [ ]:
# ============================================================
# Cell 2 — Reconstruct FIT / validation H3 slopes
# ============================================================

fit_files = sorted(V013_RUN.glob("fit-*.parquet"))
val_files = sorted(V013_RUN.glob("validation-*.parquet"))

assert len(fit_files) == 44
assert len(val_files) == 44

fit_traj = pd.concat(
    [pd.read_parquet(p) for p in fit_files],
    ignore_index=True,
)

val_traj = pd.concat(
    [pd.read_parquet(p) for p in val_files],
    ignore_index=True,
)

GROUP_COLS = [
    "query_id",
    "low",
    "high",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

def reconstruct_h3(df):
    rows = []
    for keys, g in df.groupby(GROUP_COLS, dropna=False, sort=False):
        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)
        y = g["abs_utility_gap"].to_numpy(np.float64)
        row = dict(zip(GROUP_COLS, keys))
        row["H3_abs_slope"] = float(np.polyfit(x, y, 1)[0])
        rows.append(row)
    return pd.DataFrame(rows)

fit_slopes = reconstruct_h3(fit_traj)
val_slopes = reconstruct_h3(val_traj)

assert len(fit_slopes) == 3350 * 44
assert len(val_slopes) == 3316 * 44

fit_slopes["is_amplifying_abs"] = (
    fit_slopes["H3_abs_slope"] > EPS
).astype(int)

val_slopes["is_amplifying_abs"] = (
    val_slopes["H3_abs_slope"] > EPS
).astype(int)

print("FIT rows:", fit_slopes.shape)
print("VAL rows:", val_slopes.shape)
print("FIT prevalence:", fit_slopes["is_amplifying_abs"].mean())
print("VAL prevalence:", val_slopes["is_amplifying_abs"].mean())
print("H3 TARGET RECONSTRUCTION — PASS")

del fit_traj, val_traj
gc.collect()


In [ ]:
# ============================================================
# Cell 3 — Recover the exact PQ32-only baseline feature artifact
#
# ARC-v0.15 used only:
#   query_id
#   pq32_entropy20
#   pq32_margin1_10
#
# We search persisted parquet artifacts and require one-to-one
# query coverage for all 6666 FEVER DEV queries.
# ============================================================

required_feature_cols = {
    "query_id",
    "pq32_entropy20",
    "pq32_margin1_10",
}

feature_candidates = []

# Search only under the ARC research root and keep this read-only.
for p in ARC_ROOT.rglob("*.parquet"):
    try:
        cols = set(pd.read_parquet(p, columns=[]).columns)
    except Exception:
        # Some parquet engines do not expose schema with columns=[].
        try:
            sample = pd.read_parquet(p)
            cols = set(sample.columns)
        except Exception:
            continue

    if required_feature_cols.issubset(cols):
        feature_candidates.append(p)

print("Candidate feature artifacts:", len(feature_candidates))

# Prefer smaller baseline/query feature tables over huge trajectory tables.
ranked = []
for p in feature_candidates:
    try:
        df = pd.read_parquet(
            p,
            columns=[
                "query_id",
                "pq32_entropy20",
                "pq32_margin1_10",
            ],
        )
    except Exception:
        continue

    dedup = df.drop_duplicates("query_id")
    n_unique = dedup["query_id"].nunique()

    if n_unique >= 6666:
        ranked.append(
            (
                len(df),
                p,
                dedup,
            )
        )

assert ranked, (
    "No persisted artifact with full query coverage and "
    "query_id + pq32_entropy20 + pq32_margin1_10 was found. "
    "Do not substitute SQ8/qrels-dependent features."
)

ranked.sort(key=lambda x: x[0])

_, FEATURE_SOURCE, baseline_features = ranked[0]

baseline_features = (
    baseline_features[
        [
            "query_id",
            "pq32_entropy20",
            "pq32_margin1_10",
        ]
    ]
    .drop_duplicates("query_id")
    .copy()
)

baseline_features["query_id"] = baseline_features["query_id"].astype(str)

assert baseline_features[
    [
        "pq32_entropy20",
        "pq32_margin1_10",
    ]
].notna().all().all()

print("Feature source:", FEATURE_SOURCE)
print("Unique feature queries:", baseline_features["query_id"].nunique())
print("PQ32 BASELINE FEATURE RECOVERY — PASS")


In [ ]:
# ============================================================
# Cell 4 — Build exact deployment-feasible model tables
# ============================================================

def add_policy_features(df):
    out = df.copy()
    out["query_id"] = out["query_id"].astype(str)
    out["is_softmax"] = (
        out["method"] == "softmax"
    ).astype(float)
    out["temperature_numeric"] = (
        pd.to_numeric(
            out["temperature"],
            errors="coerce",
        )
        .fillna(1.0)
    )
    out["log_k"] = np.log(
        out["k"].astype(float)
    )
    return out

fit_model = add_policy_features(
    fit_slopes.merge(
        baseline_features,
        on="query_id",
        how="left",
        validate="many_to_one",
    )
)

val_model = add_policy_features(
    val_slopes.merge(
        baseline_features,
        on="query_id",
        how="left",
        validate="many_to_one",
    )
)

FULL_DEPLOYABLE_FEATURES = [
    "pq32_entropy20",
    "pq32_margin1_10",
    "alpha",
    "log_k",
    "is_softmax",
    "temperature_numeric",
]

POLICY_ONLY_FEATURES = [
    "alpha",
    "log_k",
    "is_softmax",
    "temperature_numeric",
]

QUERY_ONLY_FEATURES = [
    "pq32_entropy20",
    "pq32_margin1_10",
]

ALPHA_ONLY_FEATURES = [
    "alpha",
]

for cols in [
    FULL_DEPLOYABLE_FEATURES,
    POLICY_ONLY_FEATURES,
    QUERY_ONLY_FEATURES,
    ALPHA_ONLY_FEATURES,
]:
    assert fit_model[cols].notna().all().all()
    assert val_model[cols].notna().all().all()

print("FIT model:", fit_model.shape)
print("VAL model:", val_model.shape)
print("MODEL TABLE CONSTRUCTION — PASS")


In [ ]:
# ============================================================
# Cell 5 — Frozen classifier specification + four ablations
#
# Match ARC-v0.15:
# LogisticRegression(
#     C=0.5,
#     max_iter=5000,
#     class_weight="balanced",
#     random_state=SEED,
# )
# ============================================================

MODEL_SPECS = {
    "alpha_only": ALPHA_ONLY_FEATURES,
    "policy_only": POLICY_ONLY_FEATURES,
    "pq32_query_only": QUERY_ONLY_FEATURES,
    "full_deployable": FULL_DEPLOYABLE_FEATURES,
}

def fit_model_for_features(features):
    clf = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            C=0.5,
            max_iter=5000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])

    X_fit = fit_model[features].to_numpy(np.float64)
    y_fit = fit_model["is_amplifying_abs"].to_numpy(int)

    clf.fit(X_fit, y_fit)

    return clf

models = {}
predictions = {}
rows = []

y_val = val_model["is_amplifying_abs"].to_numpy(int)

for name, features in MODEL_SPECS.items():
    clf = fit_model_for_features(features)
    p_val = clf.predict_proba(
        val_model[features].to_numpy(np.float64)
    )[:, 1]

    auc = float(
        roc_auc_score(
            y_val,
            p_val,
        )
    )

    ap = float(
        average_precision_score(
            y_val,
            p_val,
        )
    )

    models[name] = clf
    predictions[name] = p_val

    rows.append({
        "model": name,
        "feature_count": len(features),
        "features": ", ".join(features),
        "roc_auc": auc,
        "pr_auc": ap,
        "validation_prevalence": float(y_val.mean()),
    })

point_metrics = pd.DataFrame(rows)

display(point_metrics)

# Reproduction gate for the known ARC-v0.15 full model.
full_auc = float(
    point_metrics.loc[
        point_metrics["model"] == "full_deployable",
        "roc_auc",
    ].iloc[0]
)

print("Full deployable reconstructed AUC:", full_auc)

assert abs(full_auc - 0.7309) < 0.01, (
    "Full deployable model does not reproduce ARC-v0.15 closely enough.",
    full_auc,
)

print("ABLATION POINT ESTIMATES — PASS")


In [ ]:
# ============================================================
# Cell 6 — Query-cluster bootstrap confidence intervals
#
# Each query contributes multiple policy rows, so resample queries,
# not individual query-policy rows.
# ============================================================

rng = np.random.default_rng(SEED)

query_ids = val_model["query_id"].astype(str).to_numpy()
unique_queries = np.unique(query_ids)

row_indices_by_query = {
    q: np.flatnonzero(query_ids == q)
    for q in unique_queries
}

bootstrap_records = []

for rep in range(BOOTSTRAP_REPS):
    sampled_queries = rng.choice(
        unique_queries,
        size=len(unique_queries),
        replace=True,
    )

    sampled_rows = np.concatenate(
        [
            row_indices_by_query[q]
            for q in sampled_queries
        ]
    )

    y_b = y_val[sampled_rows]

    if len(np.unique(y_b)) < 2:
        continue

    for name, p_val in predictions.items():
        p_b = p_val[sampled_rows]

        bootstrap_records.append({
            "rep": rep,
            "model": name,
            "roc_auc": float(
                roc_auc_score(
                    y_b,
                    p_b,
                )
            ),
            "pr_auc": float(
                average_precision_score(
                    y_b,
                    p_b,
                )
            ),
        })

bootstrap_df = pd.DataFrame(
    bootstrap_records
)

ci_rows = []

for name, g in bootstrap_df.groupby("model"):
    ci_rows.append({
        "model": name,
        "roc_auc_ci_low": float(g["roc_auc"].quantile(0.025)),
        "roc_auc_ci_high": float(g["roc_auc"].quantile(0.975)),
        "pr_auc_ci_low": float(g["pr_auc"].quantile(0.025)),
        "pr_auc_ci_high": float(g["pr_auc"].quantile(0.975)),
        "bootstrap_reps": int(g["rep"].nunique()),
    })

ci_df = pd.DataFrame(ci_rows)

ablation_metrics = point_metrics.merge(
    ci_df,
    on="model",
    how="left",
    validate="one_to_one",
)

display(ablation_metrics)

ablation_metrics.to_csv(
    OUT / "v0161_deployable_predictor_ablation_metrics.csv",
    index=False,
)

print("QUERY-CLUSTER BOOTSTRAP — COMPLETE")


In [ ]:
# ============================================================
# Cell 7 — Paired query-cluster bootstrap:
# incremental value of query-specific PQ32 observables
# ============================================================

comparison_pairs = [
    ("full_deployable", "policy_only"),
    ("full_deployable", "alpha_only"),
    ("full_deployable", "pq32_query_only"),
    ("pq32_query_only", "policy_only"),
]

paired_rows = []

for better, baseline in comparison_pairs:
    g1 = bootstrap_df[
        bootstrap_df["model"] == better
    ][
        ["rep", "roc_auc", "pr_auc"]
    ].rename(columns={
        "roc_auc": "auc_better",
        "pr_auc": "ap_better",
    })

    g0 = bootstrap_df[
        bootstrap_df["model"] == baseline
    ][
        ["rep", "roc_auc", "pr_auc"]
    ].rename(columns={
        "roc_auc": "auc_base",
        "pr_auc": "ap_base",
    })

    m = g1.merge(
        g0,
        on="rep",
        how="inner",
        validate="one_to_one",
    )

    m["delta_auc"] = (
        m["auc_better"]
        - m["auc_base"]
    )

    m["delta_ap"] = (
        m["ap_better"]
        - m["ap_base"]
    )

    paired_rows.append({
        "comparison": f"{better} - {baseline}",
        "delta_auc_mean": float(m["delta_auc"].mean()),
        "delta_auc_ci_low": float(m["delta_auc"].quantile(0.025)),
        "delta_auc_ci_high": float(m["delta_auc"].quantile(0.975)),
        "delta_auc_p_le_0": float((m["delta_auc"] <= 0).mean()),
        "delta_pr_auc_mean": float(m["delta_ap"].mean()),
        "delta_pr_auc_ci_low": float(m["delta_ap"].quantile(0.025)),
        "delta_pr_auc_ci_high": float(m["delta_ap"].quantile(0.975)),
        "delta_pr_auc_p_le_0": float((m["delta_ap"] <= 0).mean()),
        "paired_reps": int(len(m)),
    })

paired_comparisons = pd.DataFrame(
    paired_rows
)

display(paired_comparisons)

paired_comparisons.to_csv(
    OUT / "v0161_paired_predictor_ablation_contrasts.csv",
    index=False,
)

print("PAIRED ABLATION CONTRASTS — COMPLETE")


In [ ]:
# ============================================================
# Cell 8 — Event capture at fixed risk budgets
# ============================================================

BUDGETS = [
    0.10,
    0.25,
    0.50,
]

total_events = int(y_val.sum())

capture_rows = []

for name, p_val in predictions.items():
    order = np.argsort(-p_val)

    for budget in BUDGETS:
        n_select = int(
            round(
                budget
                * len(y_val)
            )
        )

        selected = order[:n_select]

        events_captured = int(
            y_val[selected].sum()
        )

        capture_rows.append({
            "model": name,
            "budget": budget,
            "selected_rows": n_select,
            "events_captured": events_captured,
            "event_capture_fraction": (
                events_captured / total_events
            ),
            "selected_amplification_rate": float(
                y_val[selected].mean()
            ),
            "enrichment_vs_prevalence": float(
                y_val[selected].mean()
                / y_val.mean()
            ),
        })

capture_df = pd.DataFrame(
    capture_rows
)

display(
    capture_df
)

capture_df.to_csv(
    OUT / "v0161_risk_budget_event_capture.csv",
    index=False,
)

print("EVENT-CAPTURE ABLATION — COMPLETE")


In [ ]:
# ============================================================
# Cell 9 — Full-model standardized coefficients
# ============================================================

full_clf = models[
    "full_deployable"
]

coef_df = pd.DataFrame({
    "feature":
        FULL_DEPLOYABLE_FEATURES,

    "standardized_logistic_coefficient":
        full_clf
        .named_steps["model"]
        .coef_[0],
})

coef_df[
    "abs_coefficient"
] = (
    coef_df[
        "standardized_logistic_coefficient"
    ]
    .abs()
)

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False,
)

display(
    coef_df
)

coef_df.to_csv(
    OUT / "v0161_full_model_coefficients.csv",
    index=False,
)

print("COEFFICIENT AUDIT — COMPLETE")


In [ ]:
# ============================================================
# Cell 10 — Paper-facing interpretation gates
# ============================================================

metrics_idx = (
    ablation_metrics
    .set_index("model")
)

full_auc = float(
    metrics_idx.loc[
        "full_deployable",
        "roc_auc",
    ]
)

policy_auc = float(
    metrics_idx.loc[
        "policy_only",
        "roc_auc",
    ]
)

query_auc = float(
    metrics_idx.loc[
        "pq32_query_only",
        "roc_auc",
    ]
)

alpha_auc = float(
    metrics_idx.loc[
        "alpha_only",
        "roc_auc",
    ]
)

full_ap = float(
    metrics_idx.loc[
        "full_deployable",
        "pr_auc",
    ]
)

policy_ap = float(
    metrics_idx.loc[
        "policy_only",
        "pr_auc",
    ]
)

paper_summary = {
    "full_deployable_auc":
        full_auc,

    "policy_only_auc":
        policy_auc,

    "query_only_auc":
        query_auc,

    "alpha_only_auc":
        alpha_auc,

    "full_minus_policy_auc":
        full_auc - policy_auc,

    "full_deployable_pr_auc":
        full_ap,

    "policy_only_pr_auc":
        policy_ap,

    "full_minus_policy_pr_auc":
        full_ap - policy_ap,

    "query_specific_increment_positive":
        bool(
            full_auc
            > policy_auc
        ),
}

print(
    json.dumps(
        paper_summary,
        indent=2,
    )
)

(OUT / "v0161_paper_facing_summary.json").write_text(
    json.dumps(
        paper_summary,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print()
print("INTERPRETATION RULE")
print("-------------------")

if full_auc - policy_auc >= 0.03:
    print(
        "STRONG: PQ32 query-specific observables add substantial "
        "held-out discrimination beyond policy variables."
    )
elif full_auc - policy_auc >= 0.01:
    print(
        "MODERATE: PQ32 query-specific observables add measurable "
        "held-out discrimination beyond policy variables."
    )
elif full_auc > policy_auc:
    print(
        "SMALL: the full model improves over policy-only, but the "
        "incremental query-specific signal is limited."
    )
else:
    print(
        "NULL/NEGATIVE: policy variables explain essentially all of "
        "the full-model discrimination. Do not claim strong query-specific "
        "deployable prediction."
    )

print("PAPER-FACING ABLATION SUMMARY — COMPLETE")


In [ ]:
# ============================================================
# Cell 11 — Seal report
# ============================================================

report = {
    "status":
        "ARC_V0161_DEPLOYABLE_PREDICTOR_ABLATION_COMPLETE",

    "source_v013_run":
        str(V013_RUN),

    "source_v013_protocol_sha256":
        sha256_file(PROTOCOL_PATH),

    "source_v013_validation_report_sha256":
        sha256_file(VALIDATION_REPORT_PATH),

    "pq32_feature_source":
        str(FEATURE_SOURCE),

    "pq32_feature_source_sha256":
        sha256_file(FEATURE_SOURCE),

    "epsilon":
        EPS,

    "classifier":
        {
            "type":
                "StandardScaler + LogisticRegression",

            "C":
                0.5,

            "max_iter":
                5000,

            "class_weight":
                "balanced",

            "random_state":
                SEED,
        },

    "model_specs":
        MODEL_SPECS,

    "bootstrap_reps":
        BOOTSTRAP_REPS,

    "test_accessed":
        False,

    "interpretation_constraints": [
        (
            "This is a post-hoc feature-ablation audit over the "
            "existing ARC-v0.15 deployable predictor."
        ),
        (
            "Policy-only performance quantifies configuration-level "
            "risk information; query-only and full-model contrasts "
            "quantify additional PQ32 query-specific signal."
        ),
        (
            "No nonlinear classifier sweep is performed, and no "
            "claim of optimal prediction is made."
        ),
    ],

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = (
    OUT
    / "v0161_deployable_predictor_ablation_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(
    REPORT_PATH
)

(
    OUT
    / "V0161_REPORT_SHA256.txt"
).write_text(
    report_sha
    + "  "
    + REPORT_PATH.name
    + "\n",
    encoding="utf-8",
)

print()
print("=" * 80)
print("ARC-v0.16.1 DEPLOYABLE PREDICTOR ABLATION — PASS")
print("=" * 80)
print("Output:", OUT)
print("Report SHA-256:", report_sha)
print("Test accessed:", False)
print("=" * 80)
